# 2026 NeuroGolf Championship — Maximum Score Notebook
## Target: 10,000 / 10,000

**Core strategy (two-tier):**
1. **Tier 1 — Zero-MAC analytical ONNX** for tasks solvable by data-movement ops (Gather/Slice/Transpose).  Score ≈ 20–25 per task.
2. **Tier 2 — MDL-trained tiny networks** for complex tasks.  Score ≈ 13–15 per task.

**Key references:**
- [1] Liao & Gu. *ARC-AGI Without Pretraining* (CompressARC). NeurIPS ARC Workshop 2025. arXiv:2512.06104.
- [2] Jolicoeur-Martineau. *Less is More: Recursive Reasoning with Tiny Networks*. Samsung AI / arXiv:2510.04871, Oct 2025.
- [3] Lan, Geyer, Chemla, Katzir. *MDL Recurrent Neural Networks*. TACL 2022 (MIT Press).
- [4] Qu et al. *Automatic Joint Structured Pruning and Quantization*. CVPR/IEEE 2025.
- [5] *Pruning vs Quantization: Which is Better?* NeurIPS 2023, pp. 18518–18531.
- [6] *A MDL Approach to Regularization in Neural Networks*. NeurIPS Workshop 2025. arXiv:2505.13398.

In [ ]:
# Cell 1 — Install dependencies (uncomment if not available)
!pip install onnx==1.17.0 onnxoptimizer==0.4.2 onnxruntime-gpu==1.19.2 onnxscript cupy-cuda12x --quiet


In [ ]:
# Cell 2 — Imports
import os, sys, json, zipfile, copy, time, warnings, math
import threading
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
import multiprocessing as mp
from tqdm.auto import tqdm

import numpy as np
import cupy as cp

import torch
import torch.nn as nn
import torch.optim as optim

import onnx
import onnxruntime as ort
from onnx import helper, TensorProto, numpy_helper, shape_inference
import onnxoptimizer

warnings.filterwarnings("ignore")
if torch.cuda.is_available():
    torch.set_num_threads(1)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    torch.set_num_threads(4)
print("Imports OK")


In [ ]:
# Cell 3 — Paths & constants
# Auto-detect competition data directory (robust to different Kaggle slugs)
_data_candidates = [
    "/kaggle/input/neurogolf-2026",
    "/kaggle/input/the-2026-neurogolf-championship",
    "/kaggle/input/competitions/neurogolf-2026",
    "/kaggle/input/competitions/the-2026-neurogolf-championship",
]
DATA_DIR = next(
    (d for d in _data_candidates
     if os.path.isdir(d) and os.path.exists(os.path.join(d, "task001.json"))),
    _data_candidates[0]
)
if not os.path.isdir(DATA_DIR):
    print(f"WARNING: DATA_DIR not found: {DATA_DIR}  — task loading will fail.")
UTILS_DIR = os.path.join(DATA_DIR, "neurogolf_utils")
WORK_DIR  = "/kaggle/working"
SUB_DIR   = os.path.join(WORK_DIR, "submission")
SUB_ZIP   = os.path.join(WORK_DIR, "submission.zip")

os.makedirs(SUB_DIR, exist_ok=True)
sys.path.insert(0, UTILS_DIR)

CHANNELS = 10    # one-hot depth (colors 0-9)
GRID_H   = 30   # fixed padded height
GRID_W   = 30   # fixed padded width
OPSET    = 18   # ONNX opset (min required by torch 2.x exporter; supports negative-step Slice)
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_GPU  = (DEVICE == "cuda")

print(f"Data dir: {DATA_DIR}")
print(f"Sub dir:  {SUB_DIR}")
print(f"Device:   {DEVICE}")


In [ ]:
# Cell 4 — Grid encoding utilities (vectorised + CuPy GPU)

# ── CuPy GPU helpers ──────────────────────────────────────────────────────────
if USE_GPU:
    def _encode_grid_gpu(g_cpu):
        """Fill one-hot array on GPU via broadcast comparison.

        Returns a CuPy float32 array of shape [C, H, W].
        """
        g_gpu   = cp.asarray(g_cpu)
        one_hot = (g_gpu[None] == cp.arange(CHANNELS, dtype=cp.int32)[:, None, None]).astype(cp.float32)
        return one_hot  # [C, H, W]

    def color_mapping_valid_gpu(inp_flat_cpu, out_flat_cpu):
        """Check a consistent single-valued color mapping on GPU.

        Returns (is_valid: bool, mapping: int32 numpy[10]).

        Conflict proof: if input color c maps to both A and B (A≠B), the
        scatter writes mapping[c]=B (last-write).  The round-trip check then
        compares mapping[inp] against out at every position.  At the position
        where out==A we get mapping[c]=B≠A, so cp.all(...)==False.  No
        conflict can be silently missed.
        """
        inp = cp.asarray(inp_flat_cpu)
        out = cp.asarray(out_flat_cpu)
        mapping = cp.full(10, -1, dtype=cp.int32)
        mapping[inp] = out          # last-write-wins scatter
        is_valid = bool(cp.all(mapping[inp] == out).get())  # round-trip check
        return is_valid, mapping.get()

    def crop_match_gpu(inp_cpu, out_cpu):
        """Return uint8 mask [Hi-Ho+1, Wi-Wo+1] of valid crop positions (GPU).

        Uses a strided sliding-window view for fully vectorised comparison;
        all candidate windows are tested in a single batched GPU operation.
        """
        inp_gpu = cp.asarray(inp_cpu)
        out_gpu = cp.asarray(out_cpu)
        Ho, Wo  = out_gpu.shape
        windows = cp.lib.stride_tricks.sliding_window_view(inp_gpu, (Ho, Wo))
        # windows: [Hi-Ho+1, Wi-Wo+1, Ho, Wo]
        mask = cp.all(windows == out_gpu[None, None], axis=(-2, -1)).astype(cp.uint8)
        return mask.get()

    # Warm-up GPU kernels (triggers CUDA compilation before competition)
    try:
        _dg = np.zeros((3, 3), dtype=np.int32)
        _encode_grid_gpu(_dg)
        _v, _m = color_mapping_valid_gpu(
            np.zeros(4, dtype=np.int32), np.zeros(4, dtype=np.int32))
        # Correctness: 4x4 zeros cropped to 2x2 zeros matches all 9 positions
        _r = crop_match_gpu(
            np.zeros((4, 4), dtype=np.int32), np.zeros((2, 2), dtype=np.int32))
        assert _r.sum() == 9, f"crop_match_gpu correctness check failed: {_r.sum()}"
        # Non-matching: no position should match
        _r2 = crop_match_gpu(
            np.zeros((4, 4), dtype=np.int32), np.ones((2, 2), dtype=np.int32))
        assert _r2.sum() == 0, "crop_match_gpu false-positive detected"
        print("CuPy GPU warm-up OK")
    except Exception as _e:
        print(f"CuPy warm-up skipped: {_e}")
else:
    print("GPU not available -- pure NumPy active")


def encode_grid(grid):
    """ARC grid -> float32 one-hot tensor [1, 10, 30, 30].

    Vectorised via a single broadcast comparison (no Python loop over channels).
    """
    g    = np.asarray(grid, dtype=np.int32)
    H, W = g.shape
    # shape: [C, H, W] -- broadcast comparison across channel axis
    one_hot = (g[None] == np.arange(CHANNELS, dtype=np.int32)[:, None, None]).astype(np.float32)
    t = np.zeros((1, CHANNELS, GRID_H, GRID_W), dtype=np.float32)
    t[0, :, :H, :W] = one_hot
    return t


def decode_grid(tensor, H, W):
    """Float32 tensor [1,10,30,30] -> ARC grid H x W."""
    ch  = tensor[0]                         # [10, 30, 30]
    idx = np.argmax(ch, axis=0)[:H, :W]    # [H, W]
    return idx.tolist()


def grid_hw(grid):
    return len(grid), len(grid[0])

print("Grid utilities OK")


In [ ]:
# Cell 5 — Task loading
def load_task(task_id):
    path = os.path.join(DATA_DIR, f"task{task_id:03d}.json")
    with open(path) as f:
        return json.load(f)

def all_pairs(task):
    """Return all pairs: train + test + arc-gen."""
    pairs = []
    for split in ("train", "test", "arc-gen"):
        pairs.extend(task.get(split, []))
    return pairs

def train_pairs(task):
    return task.get("train", [])

# Quick sanity check
try:
    t = load_task(1)
    print(f"Task 001: {len(t.get('train',[]))} train, ", end="")
    print(f"{len(t.get('test',[]))} test, {len(t.get('arc-gen',[]))} arc-gen pairs")
except Exception as e:
    print(f"Could not load task (will work during competition): {e}")

In [ ]:
# Cell 6 — Cost & score estimator
#
# From competition example (verified):
#   Conv(10→10, 3×3): params=900, mem=39600B, MACs=810000
#   39600 = 900×4 (param bytes) + 1×10×30×30×4 (output activation bytes)
#   810000 = 10×10×9×30×30
#   cost = 850500 → score = 25 - ln(850500) ≈ 11.35

def score_from_cost(cost):
    if cost <= 0: return 25.0
    return max(1.0, 25.0 - math.log(max(cost, 1)))

def conv_cost(in_c, out_c, k, H=30, W=30, bias=False):
    params  = in_c * out_c * k * k + (out_c if bias else 0)
    p_bytes = params * 4
    a_bytes = 1 * out_c * H * W * 4   # output activation
    macs    = in_c * out_c * k * k * H * W
    return params, p_bytes + a_bytes, macs

# Architecture cost table
archs = {
    "Identity ONNX"          : (0,      0,        0),
    "Flip/Rotate ONNX"       : (4,      32,       0),
    "Gather permutation"     : (10,     80,       0),
    "BottleneckNet h=1"      : (20,     39680,    18000),
    "BottleneckNet h=4"      : (80,     43520,    72000),
    "Conv(10→10, 1×1)"       : (100,    36400,    90000),
    "LocalNet h=4 (3×3)"     : (400,    50560,    378000),
    "Conv(10→10, 3×3) base"  : (900,    39600,    810000),
}
print(f"{'Architecture':<28} {'Cost':>12} {'Score':>6}")
print("-"*50)
for name, (p,m,c) in archs.items():
    cost = p+m+c
    print(f"{name:<28} {cost:>12,} {score_from_cost(cost):>6.2f}")

In [ ]:
# Cell 7 — Analytical ONNX builders (zero-MAC solutions)
#
# Key insight [1]: For tasks expressible as data-movement ops,
# ONNX Gather/Slice/Transpose have 0 MACs → scores of 20–25.
# This is 6-14 points higher per task than any conv network!

def _int64_init(name, arr):
    return numpy_helper.from_array(np.array(arr, dtype=np.int64), name=name)

def _wrap(nodes, inits, out_name="output"):
    inp = helper.make_tensor_value_info("input",  TensorProto.FLOAT, [1,CHANNELS,GRID_H,GRID_W])
    out = helper.make_tensor_value_info(out_name, TensorProto.FLOAT, [1,CHANNELS,GRID_H,GRID_W])
    g   = helper.make_graph(nodes, "ng", [inp], [out], initializer=inits)
    m   = helper.make_model(g, opset_imports=[helper.make_opsetid("", OPSET)])
    m.ir_version = 8
    m   = shape_inference.infer_shapes(m)
    onnx.checker.check_model(m)
    return m

def build_identity_onnx():
    """output = input. 0 params, 0 MACs. Score = 25."""
    return _wrap([helper.make_node("Identity",["input"],["output"])], [])

def build_gather_permutation_onnx(perm):
    """Reorder channels by perm list. 10 int64 params, 0 MACs. Score ≈ 20.5."""
    idx  = _int64_init("perm", perm)
    node = helper.make_node("Gather",["input","perm"],["output"], axis=1)
    return _wrap([node], [idx])

def build_flip_onnx(axis):
    """Flip along axis 2 (height) or 3 (width). 4 int64 params, 0 MACs. Score ≈ 21.4."""
    sz = GRID_H if axis == 2 else GRID_W
    s  = _int64_init("sl_s",  [sz - 1])
    e  = _int64_init("sl_e",  [-(sz + 1)])
    a  = _int64_init("sl_a",  [axis])
    st = _int64_init("sl_st", [-1])
    node = helper.make_node("Slice",["input","sl_s","sl_e","sl_a","sl_st"],["output"])
    return _wrap([node], [s, e, a, st])

def build_compose_onnx(ops):
    """Chain of ops: 'flip_h', 'flip_v', 'transpose'."""
    nodes, inits, cur = [], [], "input"
    for i, op in enumerate(ops):
        nxt = f"t{i}" if i < len(ops)-1 else "output"
        if op in ("flip_h", "flip_v"):
            ax = 3 if op=="flip_h" else 2
            sz = GRID_W if ax==3 else GRID_H
            p  = f"op{i}_"
            s  = _int64_init(p+"s",  [sz-1])
            e  = _int64_init(p+"e",  [-(sz+1)])
            a  = _int64_init(p+"a",  [ax])
            st = _int64_init(p+"st", [-1])
            inits += [s,e,a,st]
            nodes.append(helper.make_node("Slice",
                [cur,p+"s",p+"e",p+"a",p+"st"],[nxt]))
        elif op == "transpose":
            nodes.append(helper.make_node("Transpose",[cur],[nxt],perm=[0,1,3,2]))
        cur = nxt
    return _wrap(nodes, inits)


def build_crop_onnx(r1, c1, h_out, w_out):
    """Crop input[r1:r1+h_out, c1:c1+w_out] and pad-back to 30×30. 0 MACs."""
    starts = _int64_init("crop_s", [0, 0, r1,         c1        ])
    ends   = _int64_init("crop_e", [1, CHANNELS, r1 + h_out, c1 + w_out])
    axes   = _int64_init("crop_a", [0, 1, 2, 3])
    pads   = numpy_helper.from_array(
        np.array([0, 0, 0, 0, 0, 0, GRID_H - h_out, GRID_W - w_out], dtype=np.int64),
        name="crop_p")
    slice_node = helper.make_node(
        "Slice", ["input", "crop_s", "crop_e", "crop_a"], ["cropped"])
    pad_node = helper.make_node("Pad", ["cropped", "crop_p"], ["output"])
    return _wrap([slice_node, pad_node], [starts, ends, axes, pads])



def build_scale_onnx(h_in, w_in, h_out, w_out):
    """Nearest-neighbour upscale ONNX. 0 learned params, 0 MACs.

    Slices the active h_in×w_in content from the padded 30×30 input,
    resizes to h_out×w_out using ONNX Resize (nearest/floor), then
    pads back to 30×30.  Works for any integer scale factor.
    """
    # Step 1: Slice [1,C,30,30] → [1,C,h_in,w_in]
    sl_s  = numpy_helper.from_array(np.array([0,0,0,0],              dtype=np.int64), "sc_ss")
    sl_e  = numpy_helper.from_array(np.array([1,CHANNELS,h_in,w_in], dtype=np.int64), "sc_se")
    sl_a  = numpy_helper.from_array(np.array([0,1,2,3],              dtype=np.int64), "sc_sa")
    slice_node = helper.make_node("Slice", ["input","sc_ss","sc_se","sc_sa"], ["sc_sliced"])

    # Step 2: Resize [1,C,h_in,w_in] → [1,C,h_out,w_out] nearest/floor
    roi_arr    = numpy_helper.from_array(np.empty([0], dtype=np.float32), "sc_roi")
    scales_arr = numpy_helper.from_array(np.empty([0], dtype=np.float32), "sc_scales")
    sizes_arr  = numpy_helper.from_array(
        np.array([1, CHANNELS, h_out, w_out], dtype=np.int64), "sc_sizes")
    resize_node = helper.make_node(
        "Resize",
        inputs=["sc_sliced", "sc_roi", "sc_scales", "sc_sizes"],
        outputs=["sc_resized"],
        mode="nearest",
        nearest_mode="floor",
        coordinate_transformation_mode="asymmetric",
    )

    # Step 3: Pad back to [1,C,30,30]
    pads_arr = numpy_helper.from_array(
        np.array([0,0,0,0, 0,0,GRID_H-h_out,GRID_W-w_out], dtype=np.int64), "sc_pads")
    pad_node = helper.make_node("Pad", ["sc_resized","sc_pads"], ["output"])

    return _wrap(
        [slice_node, resize_node, pad_node],
        [sl_s, sl_e, sl_a, roi_arr, scales_arr, sizes_arr, pads_arr],
    )


def build_tile_onnx(h_in, w_in, kH, kW):
    """Tile ONNX: repeat the active region kH×kW times. 0 params, 0 MACs.

    Slices [1,C,30,30] → [1,C,h_in,w_in], tiles to [1,C,h_in*kH,w_in*kW],
    then pads back to [1,C,30,30].
    """
    h_out, w_out = h_in * kH, w_in * kW

    # Step 1: Slice
    sl_s  = numpy_helper.from_array(np.array([0,0,0,0],              dtype=np.int64), "tl_ss")
    sl_e  = numpy_helper.from_array(np.array([1,CHANNELS,h_in,w_in], dtype=np.int64), "tl_se")
    sl_a  = numpy_helper.from_array(np.array([0,1,2,3],              dtype=np.int64), "tl_sa")
    slice_node = helper.make_node("Slice", ["input","tl_ss","tl_se","tl_sa"], ["tl_sliced"])

    # Step 2: Tile
    repeats   = numpy_helper.from_array(np.array([1,1,kH,kW], dtype=np.int64), "tl_rep")
    tile_node = helper.make_node("Tile", ["tl_sliced","tl_rep"], ["tl_tiled"])

    # Step 3: Pad back
    pads_arr = numpy_helper.from_array(
        np.array([0,0,0,0, 0,0,GRID_H-h_out,GRID_W-w_out], dtype=np.int64), "tl_pads")
    pad_node = helper.make_node("Pad", ["tl_tiled","tl_pads"], ["output"])

    return _wrap(
        [slice_node, tile_node, pad_node],
        [sl_s, sl_e, sl_a, repeats, pads_arr],
    )


print("Analytical builders OK")

In [ ]:
# Cell 8 — ONNX cost counter & validator

def onnx_cost(model):
    """Returns (params, mem_bytes, macs) for an ONNX model."""
    n_params, mem_bytes = 0, 0
    for init in model.graph.initializer:
        arr       = numpy_helper.to_array(init)
        n_params += arr.size
        mem_bytes += arr.nbytes

    # Build shape map for MAC counting
    shape_map = {}
    for x in model.graph.input:
        shp = [d.dim_value for d in x.type.tensor_type.shape.dim]
        shape_map[x.name] = shp
    for init in model.graph.initializer:
        shape_map[init.name] = list(numpy_helper.to_array(init).shape)
    for vi in model.graph.value_info:
        shp = [d.dim_value for d in vi.type.tensor_type.shape.dim]
        shape_map[vi.name] = shp
        # Activation memory for float tensors
        if vi.type.tensor_type.elem_type == TensorProto.FLOAT:
            sz = 1
            for s in shp: sz *= max(s,1)
            mem_bytes += sz * 4

    n_macs = 0
    for node in model.graph.node:
        if node.op_type == "Conv":
            w = shape_map.get(node.input[1])
            x = shape_map.get(node.input[0])
            if w and x and len(w)==4:
                out_c, in_c_g, kH, kW = w
                _, _, H_in, W_in = x if len(x)==4 else (1,10,30,30)
                n_macs += out_c * in_c_g * kH * kW * H_in * W_in

    return n_params, mem_bytes, n_macs


def validate_onnx(model, pairs):
    """Run ONNX on all pairs; return True iff ALL match exactly."""
    try:
        providers = (
            ["CUDAExecutionProvider", "CPUExecutionProvider"]
            if USE_GPU else ["CPUExecutionProvider"]
        )
        sess  = ort.InferenceSession(model.SerializeToString(), providers=providers)
        iname = sess.get_inputs()[0].name
        for pair in pairs:
            H, W = grid_hw(pair["output"])
            x    = encode_grid(pair["input"])
            y    = encode_grid(pair["output"])
            yp   = sess.run(None, {iname: x})[0]
            if not np.allclose(yp[:,:,:H,:W], y[:,:,:H,:W], atol=0.5):
                return False
        return True
    except Exception:
        return False

print("Cost counter & validator OK")


In [ ]:
# Cell 9 — Analytical task solvers (vectorised + new patterns)

def try_identity(task):
    if all(p["input"] == p["output"] for p in all_pairs(task)):
        return build_identity_onnx()
    return None


# ── Vectorised color-permutation inference ────────────────────────────────────
def infer_color_perm(pairs):
    """Return 10-element color mapping list, or None if not consistent.

    Uses numpy scatter + round-trip verification instead of a Python dict loop.
    O(N*H*W) with C-speed operations rather than O(N*H*W) Python iterations.
    """
    mapping = np.full(CHANNELS, -1, dtype=np.int32)
    for pair in pairs:
        inp = np.asarray(pair["input"],  dtype=np.int32).ravel()
        out = np.asarray(pair["output"], dtype=np.int32).ravel()
        if inp.shape != out.shape:
            return None
        # Scatter (last-write-wins).  If two cells with the same input color
        # map to different output colors the round-trip check catches it.
        cand = np.full(CHANNELS, -1, dtype=np.int32)
        cand[inp] = out
        if not np.all(cand[inp] == out):
            return None   # conflict detected
        # Merge into global mapping; detect cross-pair conflicts
        filled   = cand != -1
        conflict = filled & (mapping != -1) & (mapping != cand)
        if np.any(conflict):
            return None
        mapping = np.where(filled & (mapping == -1), cand, mapping)
    perm = list(range(CHANNELS))
    for k in range(CHANNELS):
        if mapping[k] != -1:
            perm[k] = int(mapping[k])
    return perm


def try_color_permutation(task):
    perm = infer_color_perm(all_pairs(task))
    if perm is None:
        return None
    model = build_gather_permutation_onnx(perm)
    return model if validate_onnx(model, all_pairs(task)) else None


# ── Fast path: single-color swap ─────────────────────────────────────────────
def try_color_replace(task):
    """Detect tasks where exactly one color maps to another (rest unchanged).

    Avoids the full infer_color_perm scan when only a single color changes --
    checked as a fast path before the general permutation solver.
    """
    pairs = all_pairs(task)
    src_c, dst_c = None, None
    for pair in pairs:
        inp = np.asarray(pair["input"],  dtype=np.int32)
        out = np.asarray(pair["output"], dtype=np.int32)
        if inp.shape != out.shape:
            return None
        diff = inp != out
        if not diff.any():
            continue
        unique_in  = np.unique(inp[diff])
        unique_out = np.unique(out[diff])
        # All changed pixels must involve exactly one src -> one dst color
        if len(unique_in) != 1 or len(unique_out) != 1:
            return None
        sc, dc = int(unique_in[0]), int(unique_out[0])
        # Unchanged pixels must truly be unchanged
        if np.any(out[~diff] != inp[~diff]):
            return None
        if src_c is None:
            src_c, dst_c = sc, dc
        elif src_c != sc or dst_c != dc:
            return None
    if src_c is None:
        return None   # all pairs identical -- handled by try_identity
    perm = list(range(CHANNELS))
    perm[src_c] = dst_c
    model = build_gather_permutation_onnx(perm)
    return model if validate_onnx(model, pairs) else None


# ── 8 dihedral group ops ──────────────────────────────────────────────────────
_DIHEDRAL = {
    "flip_h"   : ["flip_h"],
    "flip_v"   : ["flip_v"],
    "rot180"   : ["flip_h", "flip_v"],
    "transpose": ["transpose"],
    "rot90_cw" : ["transpose", "flip_h"],
    "rot90_ccw": ["transpose", "flip_v"],
    "antitrans": ["flip_h", "transpose"],
}


def try_geometric(task):
    # Must be same-size for all pairs
    for pair in all_pairs(task):
        Hi, Wi = grid_hw(pair["input"])
        Ho, Wo = grid_hw(pair["output"])
        if Hi != Ho or Wi != Wo:
            return None
    for ops in _DIHEDRAL.values():
        model = build_compose_onnx(ops)
        if validate_onnx(model, all_pairs(task)):
            return model
    return None


# ── Geometric transform + color permutation ───────────────────────────────────
def _apply_geom_np(arr, ops):
    """Apply a list of geometric ops to a 2-D numpy array."""
    for op in ops:
        if op == "flip_h":
            arr = arr[:, ::-1]
        elif op == "flip_v":
            arr = arr[::-1, :]
        elif op == "transpose":
            arr = arr.T
    return arr


def try_transpose_color_perm(task):
    """Detect a geometric transform followed by a color permutation.

    For each dihedral transform that makes input/output shapes compatible,
    checks whether infer_color_perm finds a non-trivial permutation.
    Builds a composed ONNX graph: geom ops -> Gather(channel permutation).
    """
    pairs = all_pairs(task)
    for op_name, ops in _DIHEDRAL.items():
        # Check size compatibility after the transform
        size_ok = True
        for pair in pairs:
            Hi, Wi = grid_hw(pair["input"])
            Ho, Wo = grid_hw(pair["output"])
            if op_name in ("transpose", "rot90_cw", "rot90_ccw", "antitrans"):
                if Wi != Ho or Hi != Wo:
                    size_ok = False; break
            else:
                if Hi != Ho or Wi != Wo:
                    size_ok = False; break
        if not size_ok:
            continue

        # Geometrically transform every input and check color perm
        transformed = []
        for pair in pairs:
            inp_t = _apply_geom_np(
                np.asarray(pair["input"], dtype=np.int32).copy(), ops)
            transformed.append({"input": inp_t.tolist(),
                                 "output": pair["output"]})

        perm = infer_color_perm(transformed)
        if perm is None:
            continue
        if perm == list(range(CHANNELS)):
            continue   # identity perm -- pure geometric, already handled

        # Build ONNX: chain geom ops then channel gather
        nodes, inits, cur = [], [], "input"
        for i, op in enumerate(ops):
            nxt = f"gt{i}"
            if op in ("flip_h", "flip_v"):
                ax = 3 if op == "flip_h" else 2
                sz = GRID_W if ax == 3 else GRID_H
                p  = f"gop{i}_"
                inits += [
                    _int64_init(p + "s",  [sz - 1]),
                    _int64_init(p + "e",  [-(sz + 1)]),
                    _int64_init(p + "a",  [ax]),
                    _int64_init(p + "st", [-1]),
                ]
                nodes.append(helper.make_node(
                    "Slice", [cur, p+"s", p+"e", p+"a", p+"st"], [nxt]))
            elif op == "transpose":
                nodes.append(helper.make_node(
                    "Transpose", [cur], [nxt], perm=[0, 1, 3, 2]))
            cur = nxt

        idx = _int64_init("gperm", perm)
        inits.append(idx)
        nodes.append(helper.make_node(
            "Gather", [cur, "gperm"], ["output"], axis=1))
        try:
            model = _wrap(nodes, inits)
            if validate_onnx(model, pairs):
                return model
        except Exception:
            continue
    return None


# ── Constant output ───────────────────────────────────────────────────────────
def try_constant(task):
    """Detect tasks where every output is the same fixed grid.

    Builds an ONNX Constant node (0 parameters, 0 MACs, input ignored).
    Score ~14.5 -- better than most neural solutions for truly constant tasks.
    """
    pairs = all_pairs(task)
    if not pairs:
        return None
    out0 = pairs[0]["output"]
    for pair in pairs[1:]:
        if pair["output"] != out0:
            return None
    const_val = encode_grid(out0)          # [1, 10, 30, 30] float32
    const_t   = numpy_helper.from_array(const_val, "const_out")
    node = helper.make_node("Constant", [], ["output"], value=const_t)
    try:
        return _wrap([node], [])
    except Exception:
        return None


# ── Vectorised crop detection ─────────────────────────────────────────────────
def try_crop(task):
    """Detect if output == input[r1:r1+H_out, c1:c1+W_out] for fixed r1,c1."""
    pairs = all_pairs(task)
    out_sizes = set((len(p["output"]), len(p["output"][0])) for p in pairs)
    if len(out_sizes) != 1:
        return None
    Ho, Wo = next(iter(out_sizes))

    candidates = None
    for pair in pairs:
        inp = np.asarray(pair["input"],  dtype=np.int32)
        out = np.asarray(pair["output"], dtype=np.int32)
        Hi, Wi = inp.shape
        if Hi < Ho or Wi < Wo:
            return None

        # Use CuPy GPU scanner if available, else numpy strided-view
        if USE_GPU:
            mask = crop_match_gpu(inp, out)
        else:
            windows = np.lib.stride_tricks.sliding_window_view(inp, (Ho, Wo))
            # windows: [Hi-Ho+1, Wi-Wo+1, Ho, Wo]
            mask = np.all(windows == out[None, None], axis=(-2, -1)).astype(np.uint8)

        rr, cc = np.where(mask)
        idxs   = set(zip(rr.tolist(), cc.tolist()))
        if not idxs:
            return None
        candidates = idxs if candidates is None else candidates & idxs
        if not candidates:
            return None

    r1, c1 = min(candidates)
    if Ho == GRID_H and Wo == GRID_W and r1 == 0 and c1 == 0:
        return None   # identity -- handled elsewhere
    model = build_crop_onnx(r1, c1, Ho, Wo)
    return model if validate_onnx(model, pairs) else None


# ── Vectorised scale / tile detection ────────────────────────────────────────
def try_scale(task):
    """Detect nearest-neighbour integer upscale (same size across all pairs)."""
    pairs = all_pairs(task)
    in_sizes  = set((len(p["input"]),  len(p["input"][0]))  for p in pairs)
    out_sizes = set((len(p["output"]), len(p["output"][0])) for p in pairs)
    if len(in_sizes) != 1 or len(out_sizes) != 1:
        return None
    h_in, w_in   = next(iter(in_sizes))
    h_out, w_out = next(iter(out_sizes))
    if h_out <= h_in or w_out <= w_in:
        return None
    if h_out % h_in != 0 or w_out % w_in != 0:
        return None
    if h_out > GRID_H or w_out > GRID_W:
        return None
    kH, kW = h_out // h_in, w_out // w_in
    # Batch all pairs for a single vectorised comparison
    inps = np.stack([np.array(p["input"])  for p in pairs])   # [N, h_in,  w_in]
    outs = np.stack([np.array(p["output"]) for p in pairs])   # [N, h_out, w_out]
    expected = np.repeat(np.repeat(inps, kH, axis=1), kW, axis=2)
    if not np.array_equal(expected, outs):
        return None
    model = build_scale_onnx(h_in, w_in, h_out, w_out)
    return model if validate_onnx(model, pairs) else None


def try_tile(task):
    """Detect if output is the input region tiled kH x kW times."""
    pairs = all_pairs(task)
    in_sizes  = set((len(p["input"]),  len(p["input"][0]))  for p in pairs)
    out_sizes = set((len(p["output"]), len(p["output"][0])) for p in pairs)
    if len(in_sizes) != 1 or len(out_sizes) != 1:
        return None
    h_in, w_in   = next(iter(in_sizes))
    h_out, w_out = next(iter(out_sizes))
    if h_out < h_in or w_out < w_in:
        return None
    if h_out % h_in != 0 or w_out % w_in != 0:
        return None
    if h_out > GRID_H or w_out > GRID_W:
        return None
    kH, kW = h_out // h_in, w_out // w_in
    if kH == 1 and kW == 1:
        return None   # identity -- handled elsewhere
    inps = np.stack([np.array(p["input"])  for p in pairs])
    outs = np.stack([np.array(p["output"]) for p in pairs])
    expected = np.tile(inps, (1, kH, kW))
    if not np.array_equal(expected, outs):
        return None
    model = build_tile_onnx(h_in, w_in, kH, kW)
    return model if validate_onnx(model, pairs) else None



# ── Crop-to-content (bounding box of non-zero pixels) ─────────────────────
def try_crop_to_content(task):
    """Detect if output = input cropped to bounding box of non-zero pixels.

    Handles tasks where the non-background (non-zero) region of the input is
    always at the same position and the output is exactly that region.
    Uses a consistent (r0, c0, h_out, w_out) across ALL pairs.
    """
    pairs = all_pairs(task)
    r0_g, c0_g, h_g, w_g = None, None, None, None
    for pair in pairs:
        inp = np.asarray(pair["input"],  dtype=np.int32)
        out = np.asarray(pair["output"], dtype=np.int32)
        rows = np.any(inp != 0, axis=1)
        cols = np.any(inp != 0, axis=0)
        if not rows.any() or not cols.any():
            return None
        r0, r1 = int(np.where(rows)[0][0]),  int(np.where(rows)[0][-1])
        c0, c1 = int(np.where(cols)[0][0]),  int(np.where(cols)[0][-1])
        h_out, w_out = r1 - r0 + 1, c1 - c0 + 1
        if out.shape != (h_out, w_out):
            return None
        if not np.array_equal(inp[r0:r1 + 1, c0:c1 + 1], out):
            return None
        if r0_g is None:
            r0_g, c0_g, h_g, w_g = r0, c0, h_out, w_out
        elif (r0, c0, h_out, w_out) != (r0_g, c0_g, h_g, w_g):
            return None   # crop position/size varies across pairs → not a fixed crop
    if r0_g is None:
        return None
    model = build_crop_onnx(r0_g, c0_g, h_g, w_g)
    return model if validate_onnx(model, pairs) else None


def try_analytical(task):
    """Try all analytical solvers; return cheapest valid model, or None.

    Order: cheapest / most-likely first.
    try_color_replace is a fast path before the full permutation scan.
    try_constant and try_transpose_color_perm are new Phase 5 additions.
    """
    for solver in [
        try_identity,
        try_color_replace,          # fast path (Phase 5.4)
        try_color_permutation,
        try_geometric,
        try_crop,
        try_scale,
        try_tile,
        try_crop_to_content,        # Phase 7.8
        try_constant,               # Phase 5.2
        try_transpose_color_perm,   # Phase 5.1
    ]:
        result = solver(task)
        if result is not None:
            return result
    return None

print("Analytical solvers OK")

In [ ]:
# Cell 10 — Neural network architectures
#
# Philosophy from [1,2]: minimize params via bottleneck design.
# MDL principle [3,6]: smallest correct network generalizes best.

class BottleneckNet(nn.Module):
    """1×1 conv 10→h→10.  Score@h=1: ~14.0"""
    def __init__(self, h=4):
        super().__init__()
        self.e = nn.Conv2d(10, h, 1, bias=False)
        self.d = nn.Conv2d(h, 10, 1, bias=False)
    def forward(self, x): return self.d(self.e(x))

class LocalNet(nn.Module):
    """3×3 bottleneck 10→h→10."""
    def __init__(self, h=4):
        super().__init__()
        self.e = nn.Conv2d(10, h, 3, padding=1, bias=False)
        self.d = nn.Conv2d(h, 10, 1, bias=False)
    def forward(self, x): return self.d(torch.relu(self.e(x)))

class DeepNet(nn.Module):
    """Multi-layer 3×3 + bottleneck output."""
    def __init__(self, h=8, layers=2, k=3):
        super().__init__()
        pad  = k // 2
        mods = [nn.Conv2d(10, h, k, padding=pad, bias=False), nn.ReLU()]
        for _ in range(layers - 1):
            mods += [nn.Conv2d(h, h, k, padding=pad, bias=False), nn.ReLU()]
        mods.append(nn.Conv2d(h, 10, 1, bias=False))
        self.net = nn.Sequential(*mods)
    def forward(self, x): return self.net(x)

class WideNet(nn.Module):
    """5×5 conv for tasks needing larger receptive field."""
    def __init__(self, h=8):
        super().__init__()
        self.e = nn.Conv2d(10, h, 5, padding=2, bias=False)
        self.m = nn.Conv2d(h, h, 3, padding=1, bias=False)
        self.d = nn.Conv2d(h, 10, 1, bias=False)
    def forward(self, x): return self.d(torch.relu(self.m(torch.relu(self.e(x)))))


class FlatNet(nn.Module):
    """Single 1×1 conv 10→10.  100 params, no hidden bottleneck.
    Score: ~14.0.  Learns any linear channel mixing.
    """
    def __init__(self):
        super().__init__()
        self.c = nn.Conv2d(10, 10, 1, bias=False)
    def forward(self, x): return self.c(x)


class DepthwiseSepNet(nn.Module):
    """Depthwise-separable bottleneck: DW 3×3 per channel + PW expand/project.

    Params@h=2: 90+20+20=130 vs LocalNet@h=4: 400.
    Params@h=4: 90+40+40=170 vs LocalNet@h=4: 400.
    Has spatial context from the DW 3×3 at much lower cost than LocalNet.
    Reference: Howard et al., MobileNets (arXiv:1704.04861).
    """
    def __init__(self, h=4):
        super().__init__()
        self.dw = nn.Conv2d(10, 10, 3, padding=1, groups=10, bias=False)  # 90 params
        self.e  = nn.Conv2d(10, h,  1, bias=False)
        self.d  = nn.Conv2d(h,  10, 1, bias=False)
    def forward(self, x):
        return self.d(torch.relu(self.e(torch.relu(self.dw(x)))))


class DWSDeepNet(nn.Module):
    """Two DWS stages for tasks needing larger effective receptive field.

    Stage-1: DW 3×3 → PW 10→h → PW h→10
    Stage-2: DW 3×3 (no pointwise, just spatial refinement)
    Params@h=4: 170+90 = 260 vs DeepNet@h=8_layers=2: ~1312+.
    """
    def __init__(self, h=4):
        super().__init__()
        self.dw1 = nn.Conv2d(10, 10, 3, padding=1, groups=10, bias=False)
        self.pw1 = nn.Conv2d(10, h,  1, bias=False)
        self.pw2 = nn.Conv2d(h,  10, 1, bias=False)
        self.dw2 = nn.Conv2d(10, 10, 3, padding=1, groups=10, bias=False)
    def forward(self, x):
        x = torch.relu(self.pw2(torch.relu(self.pw1(torch.relu(self.dw1(x))))))
        return self.dw2(x)

# Architecture search ladder: cheapest → most expensive
# DWS = depthwise-separable; much cheaper than LocalNet for same receptive field.
ARCH_LADDER = [
    # ── 1×1 only (no spatial context) ──────────────────────────────────────
    ("bn_h1",      lambda: BottleneckNet(h=1)),   # 20 params
    ("bn_h2",      lambda: BottleneckNet(h=2)),   # 40 params
    ("bn_h4",      lambda: BottleneckNet(h=4)),   # 80 params
    ("flat",       lambda: FlatNet()),             # 100 params (single 10→10 1×1)
    ("bn_h8",      lambda: BottleneckNet(h=8)),   # 160 params
    ("bn_h10",     lambda: BottleneckNet(h=10)),  # 200 params
    # ── Depthwise-separable (spatial context, low cost) ─────────────────────
    ("dws_h2",     lambda: DepthwiseSepNet(h=2)), # 130 params
    ("dws_h4",     lambda: DepthwiseSepNet(h=4)), # 170 params
    ("dws_h8",     lambda: DepthwiseSepNet(h=8)), # 250 params
    ("dws_deep_h4",lambda: DWSDeepNet(h=4)),      # 260 params
    ("dws_h10",    lambda: DepthwiseSepNet(h=10)),# 290 params
    ("dws_deep_h8",lambda: DWSDeepNet(h=8)),      # 340 params
    # ── Standard spatial convolutions ───────────────────────────────────────
    ("loc_h4",     lambda: LocalNet(h=4)),        # 400 params
    ("loc_h8",     lambda: LocalNet(h=8)),        # 800 params
    ("loc_h10",    lambda: LocalNet(h=10)),       # 1000 params
    ("deep_h8_2",  lambda: DeepNet(h=8, layers=2)),
    ("deep_h16_2", lambda: DeepNet(h=16,layers=2)),
    ("deep_h8_3",  lambda: DeepNet(h=8, layers=3)),
    ("wide_h8",    lambda: WideNet(h=8)),
    ("deep_h16_3", lambda: DeepNet(h=16,layers=3)),
    ("wide_h16",   lambda: WideNet(h=16)),
]

print("Neural network architectures OK")

In [ ]:
# Cell 11 — MDL training pipeline [1,3,6]
#
# MDL loss = CrossEntropy(f(X), Y) + lambda * sum(|w|)
# After training: prune |w| < alpha * max|w| and re-verify [4,5].


def augment_pairs(pairs, n_augment=8, seed=0):
    """Augment training pairs with random color permutations (vectorised).

    Uses numpy fancy indexing (perm[grid_array]) instead of per-cell Python
    list comprehensions -- approximately 10x faster for large grids / many augments.

    ARC tasks are color-agnostic: swapping all colors consistently preserves
    the structural transformation, giving the network more diverse examples.
    """
    rng       = np.random.RandomState(seed)
    augmented = list(pairs)
    for _ in range(n_augment):
        perm = rng.permutation(CHANNELS).astype(np.int32)  # int32 for consistency
        for pair in pairs:
            inp_arr = perm[np.asarray(pair["input"],  dtype=np.int32)]
            out_arr = perm[np.asarray(pair["output"], dtype=np.int32)]
            augmented.append({"input":  inp_arr.tolist(),
                              "output": out_arr.tolist()})
    return augmented


def to_tensors(pairs):
    xs = np.concatenate([encode_grid(p["input"])  for p in pairs], axis=0)
    ys = np.concatenate([encode_grid(p["output"]) for p in pairs], axis=0)
    return torch.from_numpy(xs).to(DEVICE), torch.from_numpy(ys).to(DEVICE)


def mdl_loss(model, X, Y, lam=5e-4):
    """MDL objective: CrossEntropy + L1 weight regularisation."""
    pred   = model(X)                         # [B,10,H,W]
    target = Y.argmax(dim=1)                  # [B,H,W]
    ce     = nn.functional.cross_entropy(pred, target)
    l1     = sum(p.abs().sum() for p in model.parameters())
    return ce + lam * l1


def is_correct(model, pairs):
    """True iff model matches EVERY pair exactly.

    Uses next(model.parameters()).device (not the global DEVICE constant)
    so this function is device-agnostic: it works correctly when called
    on a GPU model during training AND on a CPU model during pruning/export.
    """
    model_device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        for pair in pairs:
            H, W = grid_hw(pair["output"])
            x    = torch.from_numpy(encode_grid(pair["input"])).to(model_device)
            yp   = model(x).cpu().numpy()
            pred = np.argmax(yp[0], axis=0)[:H, :W]
            true = np.array(pair["output"])
            if not np.array_equal(pred, true):
                return False
    return True


def train_model(model, X, Y, val_data,
                max_epochs=3000, lr=1e-2, lam=5e-4, patience=30,
                check_every=50, deadline=None):
    """Train with MDL loss; early-stop on correctness.

    Parameters
    ----------
    X, Y        : pre-encoded float32 tensors on DEVICE (computed once outside seed loop)
    val_data    : list of raw ARC pairs for exact-match correctness check
    check_every : check is_correct every this many epochs (adaptive per arch)

    Changes vs original
    -------------------
    * X, Y passed in rather than re-encoded on every call      (Phase 4.1)
    * CosineAnnealingWarmRestarts replaces plain cosine --
      resets every T_0=50 epochs, period doubles each cycle;
      escapes local minima faster, fewer epochs to convergence  (Phase 4.3)
    * Adaptive check_every frequency                           (Phase 4.4)
    """
    opt = optim.Adam(model.parameters(), lr=lr)
    sch = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)
    best_loss, stall = float("inf"), 0
    model.train()
    for epoch in range(max_epochs):
        opt.zero_grad()
        loss = mdl_loss(model, X, Y, lam=lam)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
        l = loss.item()
        if l < best_loss - 1e-6:
            best_loss = l; stall = 0
        else:
            stall += 1
        if epoch % check_every == 0 and is_correct(model, val_data):
            return True
        if stall >= patience:
            break
        if deadline is not None and time.time() >= deadline:
            break
    return is_correct(model, val_data)


def prune_and_verify(model, val_data,
                     thresholds=[0.5, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01, 0.001]):
    """Magnitude pruning [4,5]: zero weights below threshold * max_weight."""
    best = copy.deepcopy(model)
    for thresh in thresholds:
        cand = copy.deepcopy(model)
        with torch.no_grad():
            for p in cand.parameters():
                if p.abs().max() > 0:
                    p[p.abs() < thresh * p.abs().max()] = 0.0
        if is_correct(cand, val_data):
            best = cand
    return best


def structured_channel_prune(model, val_data):
    """Remove dead (all-zero) output channels from BottleneckNet encoder.

    After magnitude pruning some hidden channels may be entirely zero.
    Rebuilding with fewer channels reduces parameter count and ONNX cost.
    Only applied to BottleneckNet (the most common tiny architecture).
    """
    if not isinstance(model, BottleneckNet):
        return model
    with torch.no_grad():
        alive   = model.e.weight.abs().sum(dim=(1, 2, 3)) > 0
        n_alive = int(alive.sum().item())
    if n_alive == 0 or n_alive == model.e.weight.shape[0]:
        return model   # nothing to prune
    new_m = BottleneckNet(h=n_alive).to(next(model.parameters()).device)
    with torch.no_grad():
        new_m.e.weight.data = model.e.weight.data[alive]
        new_m.d.weight.data = model.d.weight.data[:, alive, :, :]
    return new_m if is_correct(new_m, val_data) else model

print("MDL training pipeline OK")


In [ ]:
# Cell 12 — PyTorch -> ONNX export with graph optimization + INT8 quantization

import io

def export_onnx(model):
    """Export PyTorch model to ONNX with static shapes + optimizer passes."""
    model.eval()
    dummy = torch.zeros(1, CHANNELS, GRID_H, GRID_W)
    buf   = io.BytesIO()
    with torch.no_grad():
        torch.onnx.export(
            model, dummy, buf,
            opset_version       = OPSET,
            input_names         = ["input"],
            output_names        = ["output"],
            do_constant_folding = True,
            dynamic_axes        = None,   # static shapes required!
        )
    buf.seek(0)
    m = onnx.load(buf)
    m = shape_inference.infer_shapes(m)
    # Safe optimisation passes (no banned ops introduced)
    try:
        passes = ["eliminate_identity", "eliminate_nop_transpose",
                  "eliminate_nop_pad", "fuse_bn_into_conv",
                  "eliminate_unused_initializer"]
        m = onnxoptimizer.optimize(m, passes)
    except Exception:
        pass
    onnx.checker.check_model(m)
    return m


def quantize_onnx(model):
    """Apply INT8 dynamic quantization to cut weight memory by ~4x [Phase 6.1].

    Uses onnxruntime.quantization.quantize_dynamic.
    Falls back silently to the original fp32 model on any error.
    """
    try:
        from onnxruntime.quantization import quantize_dynamic, QuantType
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
            fp32_path = f.name
        with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
            int8_path = f.name
        onnx.save(model, fp32_path)
        quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8)
        q_model = onnx.load(int8_path)
        os.unlink(fp32_path)
        os.unlink(int8_path)
        onnx.checker.check_model(q_model)
        return q_model
    except Exception:
        return model   # fallback to fp32

print("ONNX exporter OK")

In [ ]:
# Cell 13 — Architecture search (smallest correct network)
#
# Phase 4 optimisations applied:
#   * Pre-encode training data once per task          (Phase 4.1)
#   * Adaptive check_every per arch size              (Phase 4.4)
#   * torch.compile(reduce-overhead) for CPU-only tiny models  (Phase 4.2)
#   * CosineAnnealingWarmRestarts                     (Phase 4.3, in train_model)
# Phase 3 optimisation:
#   * 5 seeds run sequentially on GPU (1 worker) or parallel on CPU (5 workers)


def _check_every_for(arch_name):
    """Adaptive correctness-check frequency: smaller nets converge faster."""
    if "bn_h1" in arch_name or "bn_h2" in arch_name:
        return 5     # tiny: check often
    if "deep" in arch_name or "wide" in arch_name:
        return 100   # large: each epoch is costlier
    return 50        # default


# Per-task wall-clock budget (seconds). 400 tasks × 7 s ≈ 47 min / 4 workers ≈ 12 min.
TASK_BUDGET_S = 7.0
# Heavy architectures (>~300 params): skip when budget is nearly exhausted
_HEAVY_ARCHS = {"deep_h8_2", "deep_h16_2", "deep_h8_3", "wide_h8",
                "deep_h16_3", "wide_h16"}


def solve_with_nn(task):
    """
    Try architectures from cheapest to most expensive.
    For each: MDL train x 2 seeds (parallel threads on GPU too)
    -> prune -> export ONNX.

    Speed-up changes vs original:
    * Hard per-task deadline (TASK_BUDGET_S)          — Phase 7.1
    * n_augment 16 → 4 (smaller batches)              — Phase 7.2
    * max_epochs 3000 → 200, patience 300 → 30        — Phase 7.3
    * lr 1e-2 → 5e-2 (faster convergence)             — Phase 7.4
    * seeds 5 → 2                                     — Phase 7.5
    * max_workers 1 → 2 on GPU                        — Phase 7.6
    * skip heavy archs when budget < 3 s              — Phase 7.7
    """
    deadline = time.time() + TASK_BUDGET_S

    tdp     = train_pairs(task)
    adp     = all_pairs(task)
    aug_tdp = augment_pairs(tdp, n_augment=4)

    # Pre-encode training data ONCE (Phase 4.1)
    X, Y = to_tensors(aug_tdp)  # tensors are on DEVICE

    for arch_name, arch_fn in ARCH_LADDER:
        # Phase 7.1: bail out when budget is exhausted
        remaining = deadline - time.time()
        if remaining <= 0:
            break
        # Phase 7.7: skip expensive archs when < 3 s remain
        if arch_name in _HEAVY_ARCHS and remaining < 3.0:
            continue

        max_epochs  = 200
        check_every = _check_every_for(arch_name)

        found_model = [None]
        result_lock = threading.Lock()

        def _train_one_seed(seed, _arch_fn=arch_fn, _arch_name=arch_name,
                             _X=X, _Y=Y, _adp=adp,
                             _max_epochs=max_epochs, _check_every=check_every,
                             _deadline=deadline):
            """Train one seed; store result if correct and no other seed won."""
            with result_lock:
                if found_model[0] is not None:
                    return   # another seed already solved it

            # Phase 7.1: abort seed if deadline already passed
            if time.time() >= _deadline:
                return

            # Thread-safe weight init via per-thread numpy RNG (avoids
            # torch.manual_seed global RNG races between threads)
            rng = np.random.RandomState(seed * 42)
            m   = _arch_fn().to(DEVICE)   # move model to GPU if available
            with torch.no_grad():
                for p in m.parameters():
                    if p.dim() >= 2:
                        # Xavier uniform: Uniform(-a, a),
                        # a = sqrt(6 / (fan_in + fan_out))
                        fan_in  = p[0].numel()
                        fan_out = p.shape[0] * max(p[0].numel() // p.shape[1], 1)
                        a = np.sqrt(6.0 / max(fan_in + fan_out, 1))
                        p.copy_(torch.from_numpy(
                            rng.uniform(-a, a, tuple(p.shape)).astype(np.float32)))
                    else:
                        p.zero_()

            # torch.compile for reduced dispatch overhead (CPU tiny models).
            m_run = m
            if not USE_GPU and ("bn" in _arch_name or "flat" in _arch_name):
                try:
                    m_run = torch.compile(m, mode="reduce-overhead")
                except Exception:
                    pass   # torch.compile unavailable -- use eager mode

            ok = train_model(m_run, _X, _Y, _adp,
                             max_epochs=_max_epochs, lr=5e-2, lam=5e-4,
                             check_every=_check_every, deadline=_deadline)
            if ok:
                with result_lock:
                    if found_model[0] is None:
                        # Store on CPU for pruning / ONNX export compatibility
                        found_model[0] = m.cpu()

        # Phase 7.6: GPU now uses 2 parallel threads (small models fit easily).
        # CPU: 2 seeds in parallel via threads.
        max_workers = 2
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            list(ex.map(_train_one_seed, range(2)))

        if found_model[0] is not None:
            model      = prune_and_verify(found_model[0], adp)
            model      = structured_channel_prune(model, adp)
            onnx_model = export_onnx(model.cpu())   # must be on CPU for ONNX export
            if validate_onnx(onnx_model, adp):
                return onnx_model

    return None

print("Architecture search OK")


In [ ]:
# Cell 14 — Per-task processor

def process_task(task_id, verbose=True):
    """
    Solve one task; save .onnx to SUB_DIR; return info dict.

    Changes vs original:
    * On CPU mode: sets torch num_threads to 1 when running inside a subprocess
      (task parallelism in Cell 15) to avoid CPU over-subscription.
    * Applies INT8 quantization to neural ONNX models (Phase 6.1).
    """
    # Limit intra-op threads when running inside a CPU worker process
    if not USE_GPU and mp.current_process().name != "MainProcess":
        torch.set_num_threads(1)

    t0   = time.time()
    task = load_task(task_id)

    # Tier 1: analytical (zero-MAC)
    model  = try_analytical(task)
    method = "analytical"

    # Tier 2: MDL-trained tiny network
    if model is None:
        model  = solve_with_nn(task)
        method = "neural"

    out_path = os.path.join(SUB_DIR, f"task{task_id:03d}.onnx")

    if model is not None and isinstance(model, onnx.ModelProto):
        # Phase 6.1: apply INT8 quantization to neural models to reduce cost
        if method == "neural":
            model = quantize_onnx(model)
        onnx.save(model, out_path)
        p, m, c = onnx_cost(model)
    else:
        # Absolute fallback: identity passthrough
        fallback = build_identity_onnx()
        onnx.save(fallback, out_path)
        p, m, c  = 0, 0, 0
        method   = "fallback"

    cost    = p + m + c
    score   = score_from_cost(cost)
    elapsed = time.time() - t0

    if verbose:
        print(f"Task {task_id:03d} | {method:<10} | "
              f"p={p:>6,} m={m:>7,}B macs={c:>9,} | "
              f"cost={cost:>10,} score={score:>5.2f} | {elapsed:.1f}s")

    return dict(task_id=task_id, method=method,
                params=p, memory=m, macs=c,
                cost=cost, score=score, elapsed=elapsed)

print("Per-task processor OK")


In [ ]:
# Cell 15 — Main loop: process all 400 tasks
# Speed-up edition: target ~10-20 min total (TASK_BUDGET_S + 4 workers).
#
# GPU mode : ThreadPoolExecutor (CUDA context is not fork-safe).
#   * max_workers=4: tiny models leave ample GPU headroom per slot.
# CPU mode : ProcessPoolExecutor via fork.
#   * max_workers=8: over-subscribe slightly; I/O pauses keep cores busy.
#   * Each worker calls torch.set_num_threads(1) (done inside process_task)
#     to avoid 8 tasks x N threads competing for the same cores.

N_TASKS     = 400   # change to e.g. 5 for a quick smoke-test
MAX_WORKERS = 4 if USE_GPU else 8

results     = []
total_score = 0.0
n_crashed   = 0

if USE_GPU:
    # GPU mode: threads share the CUDA context safely
    _executor_cls    = ThreadPoolExecutor
    _executor_kwargs = {"max_workers": MAX_WORKERS}
else:
    # CPU mode: use fork-based processes for true parallelism
    _mp_ctx          = mp.get_context("fork")
    _executor_cls    = ProcessPoolExecutor
    _executor_kwargs = {"max_workers": MAX_WORKERS, "mp_context": _mp_ctx}

with tqdm(total=N_TASKS, desc="Tasks", unit="task",
          dynamic_ncols=True, smoothing=0.1) as pbar:
    with _executor_cls(**_executor_kwargs) as pool:
        futures = {pool.submit(process_task, tid, False): tid
                   for tid in range(1, N_TASKS + 1)}

        completed = 0
        for fut in as_completed(futures):
            tid  = futures[fut]
            try:
                info = fut.result()
            except Exception as exc:
                # Hard fallback: record a score entry even if the worker crashed.
                tqdm.write(f"Task {tid:03d} CRASHED: {exc}")
                info = dict(task_id=tid, method="fallback",
                            params=0, memory=0, macs=0, cost=0, score=1.0, elapsed=0)
                n_crashed += 1
            results.append(info)
            total_score += info["score"]
            completed   += 1

            avg  = total_score / completed
            proj = avg * N_TASKS
            postfix = dict(score=f"{total_score:.1f}",
                           avg=f"{avg:.2f}/25",
                           proj=f"{proj:.0f}/10000")
            if n_crashed:
                postfix["crashed"] = n_crashed
            pbar.set_postfix(postfix)
            pbar.update(1)

            # Milestone log every 100 tasks for a permanent scrollback record
            if completed % 100 == 0:
                tqdm.write(f"-- {completed}/{N_TASKS} done | "
                           f"total={total_score:.1f} | avg={avg:.2f}/25 | "
                           f"proj={proj:.0f}/10000 --")

# Final summary line after the bar closes
avg  = total_score / N_TASKS if results else 0.0
proj = avg * N_TASKS
print(f"\nFINAL: {N_TASKS} tasks | total={total_score:.1f} | "
      f"avg={avg:.2f}/25 | proj={proj:.0f}/10000"
      + (f" | crashed={n_crashed}" if n_crashed else ""))

# Sort results by task_id for the summary
results.sort(key=lambda r: r["task_id"])


In [ ]:
# Cell 16 — Build submission.zip

import zipfile

def build_submission():
    with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for tid in range(1, 401):
            path = os.path.join(SUB_DIR, f"task{tid:03d}.onnx")
            if os.path.exists(path):
                zf.write(path, arcname=f"task{tid:03d}.onnx")

    total  = sum(r["score"]              for r in results)
    analy  = sum(1 for r in results if r["method"] == "analytical")
    neural = sum(1 for r in results if r["method"] == "neural")
    fall   = sum(1 for r in results if r["method"] == "fallback")

    print(f"{'='*62}")
    print(f"  Submission file:       {SUB_ZIP}")
    print(f"  Estimated total score: {total:.1f} / 10,000")
    print(f"  Analytical solutions:  {analy}/400  (score ≈ 20+)")
    print(f"  Neural solutions:      {neural}/400  (score ≈ 11-15)")
    print(f"  Fallback (identity):   {fall}/400")
    print(f"  Average score/task:    {total/max(len(results),1):.2f} / 25.00")
    print(f"{'='*62}")
    return SUB_ZIP

path = build_submission()
print(f"\nReady to submit: {path}")